## Two Stage Hyperparameter Optimization

### Stage 1: uf (satisfiable) instances
- Fixed Params

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path
import torch
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import argparse
from eval import maxsat_dNN_single_cnf   # or your actual import
import optuna.logging
optuna.logging.set_verbosity(optuna.logging.INFO)
import shutil
# ----------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------
TUNING_DIR = Path("instances/uf100.430.1000")
CNF_FILES = sorted(TUNING_DIR.glob("*.cnf"))[:100]   # first 100 instances

DEVICE = torch.device("cpu")
DTYPE = torch.float32
N_TRIALS = 100          
N_WARMUP_TRIALS = 20  
TIMEOUT = None        

FIXED_ARGS = {
    "epochs": 1000,        
    "grad_clip": 1.0,
    "eta_min": 0,
    "warmup_epochs": 10,
    "restart_samples": 20,
    "confidence_threshold": 0.05,
    "max_free_vars": 20,
    "random_noise": 0.05,
    "max_free_vars": 20,

}

def objective(trial: optuna.Trial) -> float:
    """Run all tuning instances with the trial's hyperparameters, return mean best_objective."""

   
    batch_size = trial.suggest_int("batch_size", 1, 64, log=False)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-1, log=True)
    restart_period = trial.suggest_int('restart_period', 10, 200, step=10)
    patience = trial.suggest_int('patience', 5, 100, step=5)

    args = argparse.Namespace(
        epochs=FIXED_ARGS["epochs"],
        lr=lr,
        batch_size=batch_size,
        grad_clip=FIXED_ARGS["grad_clip"],
        eta_min=FIXED_ARGS["eta_min"],
        patience=patience,
        weight_decay=weight_decay,
        random_noise=FIXED_ARGS["random_noise"],
        warmup_epochs=FIXED_ARGS["warmup_epochs"],
        restart_period=restart_period,
        restart_samples=FIXED_ARGS["restart_samples"],
        confidence_threshold=FIXED_ARGS["confidence_threshold"],
        max_free_vars=FIXED_ARGS["max_free_vars"],
    )

    # Use a temporary output directory to avoid cluttering disk;
    # we don't need per‑instance plots during tuning.
    output_dir = Path(f"optuna_tmp/trial_{trial.number}")
    output_dir.mkdir(parents=True, exist_ok=True)

    objectives = []
    for i, cnf_file in enumerate(CNF_FILES):
        result = maxsat_dNN_single_cnf(
            cnf_path=str(cnf_file),
            args=args,
            output_dir=output_dir,
            device=DEVICE,
            dtype=DTYPE,
        )
        objectives.append(result["best_objective"])
        trial.report(sum(objectives) / len(objectives), step=i)
        if trial.should_prune():
            shutil.rmtree(output_dir, ignore_errors=True)
            raise optuna.TrialPruned()

    # Clean up temporary directory (we'll re‑run the best one later)
    shutil.rmtree(output_dir, ignore_errors=True)
    return sum(objectives) / len(objectives)

study = optuna.create_study(
    study_name="maxsat_test",
    storage="sqlite:///optuna_study.db",
    direction="minimize",
    sampler=TPESampler(n_startup_trials=N_WARMUP_TRIALS),
    pruner=MedianPruner(n_startup_trials=N_WARMUP_TRIALS, n_warmup_steps=5),
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT, show_progress_bar=True)
# ----------------------------------------------------------------------
# 3. Save best parameters
# ----------------------------------------------------------------------
best_params = study.best_params
best_value = study.best_value

print("Best objective (mean best_objective):", best_value)
print("Best hyperparameters:", json.dumps(best_params, indent=2))

# Save to a JSON file for Stage 2
with open("optuna_tmp/best_params_stage1.json", "w") as f:
    json.dump(best_params, f, indent=2)
    
# Re‑run the solver with the best parameters, this time keeping the logs
best_output_dir = Path("optuna_tmp/best_trial")
best_output_dir.mkdir(parents=True, exist_ok=True)
best_args = argparse.Namespace(**best_params)

for cnf_file in CNF_FILES:
    maxsat_dNN_single_cnf(
        cnf_path=str(cnf_file),
        args=best_args,
        output_dir=best_output_dir,
        device=DEVICE,
        dtype=DTYPE,
    )

print(f"Best trial output saved in {best_output_dir}")

[I 2026-06-14 21:57:33,117] Using an existing study with name 'maxsat_test' instead of creating a new one.


  0%|          | 0/100 [00:00<?, ?it/s]

/Users/aeg00011/MaxSAT/helpers.py:75: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


[I 2026-06-14 21:58:03,710] Trial 18 finished with value: 1692.6356243896485 and parameters: {'batch_size': 53, 'lr': 0.098431861317463, 'weight_decay': 0.00020790245499837314, 'restart_period': 50, 'patience': 20}. Best is trial 18 with value: 1692.6356243896485.
[I 2026-06-14 21:59:30,787] Trial 19 finished with value: 3329.027490234375 and parameters: {'batch_size': 40, 'lr': 0.0010406532945372307, 'weight_decay': 1.101256446436969e-06, 'restart_period': 120, 'patience': 5}. Best is trial 18 with value: 1692.6356243896485.
[I 2026-06-14 22:01:10,232] Trial 20 finished with value: 3242.054235839844 and parameters: {'batch_size': 57, 'lr': 0.0009942498590876944, 'weight_decay': 1.5024131754830452e-05, 'restart_period': 10, 'patience': 60}. Best is trial 18 with value: 1692.6356243896485.
[I 2026-06-14 22:01:21,523] Trial 21 finished with value: 1948.7672241210937 and parameters: {'batch_size': 4, 'lr': 0.08353269837935215, 'weight_decay': 8.904158606896375e-06, 'restart_period': 80, '